In [1]:
# import packages 
import pandas as pd
import re
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.inspection import permutation_importance
from sklearn.model_selection import StratifiedKFold

In [2]:
# load UPDATED STYLOMETRIC DATA (50 line Ovid dataset)
data = pd.read_csv('20251023_034606-normed.csv')

# remove whitespace from references
data['Locus'] = data['Locus'].str.replace(r'.txt', '')
data['Locus'] = data['Locus'].apply(lambda x: re.sub(r'\s+(\d{3})', r'\1', x))
data.head()

,Locus,word_count,sentence_count,sentence_length,fraction_sentence_relative,relative_clause_length,alius,antequam,atque_consonant,conjunction,...,personal,preposition,priusquam,quidam,quin,quominus,reflexive,si,superlative,ut
0,1.182-1.198,113,6,18.833333,0.666667,5.333333,0.0,0,0.0,0.097345,...,0.044248,0.044248,0.0,0.0,0.0,0,0.0,0.000,0.0,0.0
1,1.209-1.243,230,17,13.529412,0.294118,5.500000,0.0,0,0.0,0.065217,...,0.008696,0.034783,0.0,0.0,0.0,0,0.0,0.000,0.0,0.0
2,1.456-1.462,46,3,15.333333,0.666667,7.333333,0.0,0,0.0,0.043478,...,0.043478,0.000000,0.0,0.0,0.0,0,0.0,0.000,0.0,0.0
3,1.498-1.498,8,3,2.666667,0.000000,NaN,0.0,0,0.0,0.250000,...,0.000000,0.000000,0.0,0.0,0.0,0,0.0,0.125,0.0,0.0
4,1.504-1.524,149,13,11.461538,0.461538,3.857143,0.0,0,0.0,0.087248,...,0.060403,0.026846,0.0,0.0,0.0,0,0.0,0.000,0.0,0.0


In [3]:
# load class labels 
labels = pd.read_excel('20240830_071623-normed.xlsx',sheet_name='labels')

#remove whitespace from references
labels['Text'] = labels['Text'].apply(lambda x: re.sub(r'\s+(\d{3})', r'\1', x))
labels.head()

,Text,Locus,Speaker,Addressee,Human/Divine,Male/Female
0,Ovid Metamorphoses,1.182-1.198,Jupiter,gods,Divine,Male
1,Ovid Metamorphoses,1.209-1.243,Jupiter,gods,Divine,Male
2,Ovid Metamorphoses,1.456-1.462,Apollo,Cupid,Divine,Male
3,Ovid Metamorphoses,1.456-1.462,Apollo,Cupid,Divine,Male
4,Ovid Metamorphoses,1.498-1.498,Apollo,Apollo,Divine,Male


In [4]:
# merge stylometry data with labels using references
combined_data = pd.merge(data, labels, on='Locus')

# drop duplicates and speeches with less than 25 words 
combined_data = combined_data.drop_duplicates(subset=['Locus'],keep='last')
combined_data = combined_data[combined_data.word_count > 20]
combined_data.head()

,Locus,word_count,sentence_count,sentence_length,fraction_sentence_relative,relative_clause_length,alius,antequam,atque_consonant,conjunction,...,quominus,reflexive,si,superlative,ut,Text,Speaker,Addressee,Human/Divine,Male/Female
0,1.182-1.198,113,6,18.833333,0.666667,5.333333,0.0,0,0.0,0.097345,...,0,0.0,0.000000,0.000000,0.0,Ovid Metamorphoses,Jupiter,gods,Divine,Male
1,1.209-1.243,230,17,13.529412,0.294118,5.500000,0.0,0,0.0,0.065217,...,0,0.0,0.000000,0.000000,0.0,Ovid Metamorphoses,Jupiter,gods,Divine,Male
3,1.456-1.462,46,3,15.333333,0.666667,7.333333,0.0,0,0.0,0.043478,...,0,0.0,0.000000,0.000000,0.0,Ovid Metamorphoses,Apollo,Cupid,Divine,Male
5,1.504-1.524,149,13,11.461538,0.461538,3.857143,0.0,0,0.0,0.087248,...,0,0.0,0.000000,0.000000,0.0,Ovid Metamorphoses,Apollo,Daphne,Divine,Male
6,1.589-1.597,61,4,15.250000,0.500000,4.333333,0.0,0,0.0,0.147541,...,0,0.0,0.016393,0.016393,0.0,Ovid Metamorphoses,Jupiter,Io,Divine,Male


In [5]:
#merge data
#drop word and sentence counts
combined_data = combined_data.drop(['word_count', 'sentence_count'], axis=1)

# recode class labels
combined_data['Human/Divine'] = combined_data['Human/Divine'].replace(to_replace=['Divine', 'Human'], value=[0, 1])
combined_data['Male/Female'] = combined_data['Male/Female'].replace(to_replace=['Female', 'Male'], value=[0, 1])

# male and female subsets
female_data = combined_data.loc[combined_data['Male/Female'] == 0]
male_data = combined_data.loc[combined_data['Male/Female'] == 1]

# export modified sheets
combined_data.to_excel('data_RF.xlsx')
female_data.to_excel('female.xlsx')
male_data.to_excel('male.xlsx')

combined_data.head()

,Locus,sentence_length,fraction_sentence_relative,relative_clause_length,alius,antequam,atque_consonant,conjunction,cum_clause,demonstrative,...,quominus,reflexive,si,superlative,ut,Text,Speaker,Addressee,Human/Divine,Male/Female
0,1.182-1.198,18.833333,0.666667,5.333333,0.0,0,0.0,0.097345,0.0,0.026549,...,0,0.0,0.000000,0.000000,0.0,Ovid Metamorphoses,Jupiter,gods,0,1
1,1.209-1.243,13.529412,0.294118,5.500000,0.0,0,0.0,0.065217,0.0,0.026087,...,0,0.0,0.000000,0.000000,0.0,Ovid Metamorphoses,Jupiter,gods,0,1
3,1.456-1.462,15.333333,0.666667,7.333333,0.0,0,0.0,0.043478,0.0,0.021739,...,0,0.0,0.000000,0.000000,0.0,Ovid Metamorphoses,Apollo,Cupid,0,1
5,1.504-1.524,11.461538,0.461538,3.857143,0.0,0,0.0,0.087248,0.0,0.013423,...,0,0.0,0.000000,0.000000,0.0,Ovid Metamorphoses,Apollo,Daphne,0,1
6,1.589-1.597,15.250000,0.500000,4.333333,0.0,0,0.0,0.147541,0.0,0.000000,...,0,0.0,0.016393,0.016393,0.0,Ovid Metamorphoses,Jupiter,Io,0,1


In [6]:
combined_data

,Locus,sentence_length,fraction_sentence_relative,relative_clause_length,alius,antequam,atque_consonant,conjunction,cum_clause,demonstrative,...,quominus,reflexive,si,superlative,ut,Text,Speaker,Addressee,Human/Divine,Male/Female
0,1.182-1.198,18.833333,0.666667,5.333333,0.000000,0,0.0,0.097345,0.000000,0.026549,...,0,0.000000,0.000000,0.000000,0.000000,Ovid Metamorphoses,Jupiter,gods,0,1
1,1.209-1.243,13.529412,0.294118,5.500000,0.000000,0,0.0,0.065217,0.000000,0.026087,...,0,0.000000,0.000000,0.000000,0.000000,Ovid Metamorphoses,Jupiter,gods,0,1
3,1.456-1.462,15.333333,0.666667,7.333333,0.000000,0,0.0,0.043478,0.000000,0.021739,...,0,0.000000,0.000000,0.000000,0.000000,Ovid Metamorphoses,Apollo,Cupid,0,1
5,1.504-1.524,11.461538,0.461538,3.857143,0.000000,0,0.0,0.087248,0.000000,0.013423,...,0,0.000000,0.000000,0.000000,0.000000,Ovid Metamorphoses,Apollo,Daphne,0,1
6,1.589-1.597,15.250000,0.500000,4.333333,0.000000,0,0.0,0.147541,0.000000,0.000000,...,0,0.000000,0.016393,0.016393,0.000000,Ovid Metamorphoses,Jupiter,Io,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117,9.4-9.88,15.583333,0.222222,7.000000,0.003559,0,0.0,0.110320,0.001779,0.016014,...,0,0.003559,0.000000,0.001779,0.003559,Ovid Metamorphoses,Acheloüs,Theseus,0,1
118,9.474-9.519,9.171429,0.228571,3.875000,0.000000,0,0.0,0.102804,0.000000,0.037383,...,0,0.006231,0.021807,0.000000,0.006231,Ovid Metamorphoses,Byblis,Byblis,1,0
120,9.530-9.563,19.916667,0.333333,4.600000,0.000000,0,0.0,0.138075,0.000000,0.008368,...,0,0.000000,0.016736,0.004184,0.004184,Ovid Metamorphoses,Byblis,Caunus,1,0
122,9.585-9.629,13.217391,0.304348,5.428571,0.000000,0,0.0,0.154605,0.000000,0.026316,...,0,0.000000,0.013158,0.000000,0.006579,Ovid Metamorphoses,Byblis,Byblis,1,0


In [7]:
# train random forest classifier

In [8]:
# stylometric features
#features = combined_data.loc[:, 'alius':'ut']
features = combined_data.loc[:, 'sentence_length':'ut']

In [12]:
feature_columns = features
X = feature_columns
y = combined_data['Male/Female']

# initialize Random Forest Classifier
rf = RandomForestClassifier(n_estimators=1000)

# initialize StratifiedKFold
skf = StratifiedKFold(n_splits=5)

# perform 5-fold stratified cross-validation
cv_scores = cross_val_score(rf, X, y, cv=skf)

# F1 scores
cv_scores_f1 = cross_val_score(rf, X, y, scoring='f1')

# output the results
print("Stratified CV Scores:", cv_scores)
print("Mean Stratified CV Score:", np.mean(cv_scores))
print("CV F1 Scores:", cv_scores_f1)
print("Mean CV F1 Scores:", np.mean(cv_scores_f1))

Stratified CV Scores: [0.52941176 0.47058824 0.64705882 0.70588235 0.58823529]
Mean Stratified CV Score: 0.5882352941176471
CV F1 Scores: [0.54545455 0.4        0.7826087  0.70588235 0.63157895]
Mean CV F1 Scores: 0.6131049082832634


In [13]:
# train the model on the entire dataset
rf.fit(X, y)

# gini importance
gini_importances = rf.feature_importances_

# permutation importance
perm_importance = permutation_importance(rf, X, y, n_repeats=50)

# combine feature names with their Gini and permutation importances
gini_feature_importances = [(feature, importance) for feature, importance in zip(feature_columns, gini_importances)]
perm_feature_importances = [(feature, importance) for feature, importance in zip(feature_columns, perm_importance.importances_mean)]

# sort the features by their importances
sorted_gini_importances = sorted(gini_feature_importances, key=lambda x: x[1], reverse=True)
sorted_perm_importances = sorted(perm_feature_importances, key=lambda x: x[1], reverse=True)

# output the sorted importances
print("Sorted Gini Importances:")
for feature, importance in sorted_gini_importances:
    print(f"{feature}: {importance}")

print("\nSorted Permutation Importances:")
for feature, importance in sorted_perm_importances:
    print(f"{feature}: {importance}")

Sorted Gini Importances:
conjunction: 0.09944943505842296
relative_clause_length: 0.09548568119191986
sentence_length: 0.08428588896795351
demonstrative: 0.08209366208552787
interrogative: 0.0803307890934163
preposition: 0.0653420005072702
personal: 0.06400838074258966
si: 0.05830511398200588
fraction_sentence_relative: 0.0486028695794428
reflexive: 0.045316873908777255
idem: 0.034556080742408915
gerundive: 0.03442832449914533
superlative: 0.03180555435874745
ut: 0.03019183649581739
o_interjection: 0.030123601021012296
ipse: 0.030008172518029324
iste: 0.024983394403834864
cum_clause: 0.01939303184347789
dum: 0.018686184445812607
alius: 0.013425217823738195
atque_consonant: 0.0046335207324035245
quin: 0.002098657593994447
quidam: 0.0019343193642505552
priusquam: 0.0005114090400009816
antequam: 0.0
quominus: 0.0

Sorted Permutation Importances:
relative_clause_length: 0.019294117647058816
conjunction: 0.01294117647058821
interrogative: 0.010588235294117619
sentence_length: 0.002588235294